# African Bat Occurrence Records with Taxonomy, Collection Metadata, and Geographic Coordinates Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, inspecting, and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.hdaq-ptfb/fair2.json`

This dataset contains African bat occurrence records with taxonomic, geographic, temporal, specimen, and provenance-related metadata.

> **Important note**: In Croissant datasets, record sets, fields, columns, and related entities should be referenced by their `@id` values. This notebook uses variables and metadata-driven discovery where possible, and it works safely even if some record sets are absent from the schema metadata.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn pyarrow

## 1. Data Loading
Load the dataset metadata from the Croissant schema using `mlcroissant`, then inspect high-level dataset information.

We keep the dataset URL in a variable so it can be reused throughout the notebook.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.hdaq-ptfb/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully.")
print(f"Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"URL: {croissant_url}")

In [ ]:
# Convert metadata to a JSON-like object for inspection if needed
# This is useful for exploring the full Croissant structure.
metadata_json = metadata.to_json()

print("Top-level metadata keys:")
print(sorted(metadata_json.keys()))

## 2. Data Overview
Review the dataset metadata, available record sets, and any discoverable fields.

This dataset package summary indicates a parquet-backed occurrence table, but the provided package excerpt shows an empty `recordSet` list. Because Croissant metadata can vary in how fully record sets are exposed, this section first inspects the metadata programmatically and then attempts safe record access.

All discovered entities are displayed using their `@id` values when available.

In [ ]:
# Inspect selected top-level metadata fields
summary_fields = [
    '@id', 'name', 'description', 'identifier', 'citeAs', 'datePublished',
    'spatialCoverage', 'temporalCoverage', 'keywords', 'distribution', 'recordSet'
]

for field_name in summary_fields:
    value = metadata_json.get(field_name, None)
    print(f"\n{field_name}:")
    pprint(value)

In [ ]:
# Discover record set @ids from metadata
record_sets_metadata = metadata_json.get('recordSet', []) or []
record_set_ids = []

print("Discovered record sets:")
if not record_sets_metadata:
    print("- No record sets were explicitly listed in metadata_json['recordSet'].")
else:
    for rs in record_sets_metadata:
        rs_id = rs.get('@id', 'UNKNOWN_RECORD_SET_ID') if isinstance(rs, dict) else str(rs)
        record_set_ids.append(rs_id)
        print(f"- {rs_id}")

In [ ]:
# If record sets are present, inspect their fields by @id.
if record_sets_metadata:
    for rs in record_sets_metadata:
        rs_id = rs.get('@id', 'UNKNOWN_RECORD_SET_ID')
        print(f"\nRecord set @id: {rs_id}")
        print(f"Name: {rs.get('name', 'N/A')}")
        print("Fields:")
        for field in rs.get('field', []) or []:
            field_id = field.get('@id', 'UNKNOWN_FIELD_ID') if isinstance(field, dict) else str(field)
            field_name = field.get('name', 'N/A') if isinstance(field, dict) else 'N/A'
            field_type = field.get('dataType', 'N/A') if isinstance(field, dict) else 'N/A'
            print(f"  - @id: {field_id} | name: {field_name} | dataType: {field_type}")
else:
    print("No explicit record-set field definitions were available to inspect.")

### Attempting record access
The template expects records to be loaded from a record set using `dataset.records(record_set=<record_set_id>)`.

Because this dataset excerpt does not expose record-set `@id`s directly, the next cell tries two safe strategies:
1. Use discovered record-set `@id`s if present.
2. If none are exposed, try the generic `dataset.records()` interface and inspect a small sample.

This keeps the notebook robust and still aligned with Croissant-first exploration.

In [ ]:
sample_records = []
sample_record_set_id = None

if record_set_ids:
    sample_record_set_id = record_set_ids[0]
    print(f"Trying dataset.records(record_set={sample_record_set_id!r}) ...")
    try:
        for i, x in enumerate(dataset.records(record_set=sample_record_set_id)):
            sample_records.append(x)
            print(x)
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not read records from record set {sample_record_set_id}: {e}")
else:
    print("No record-set @id discovered from metadata. Trying dataset.records() without an explicit record_set argument...")
    try:
        for i, x in enumerate(dataset.records()):
            sample_records.append(x)
            print(x)
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not read records using dataset.records(): {e}")

## 3. Data Extraction
Load records into pandas DataFrames for analysis.

If record-set `@id`s are available, we extract one DataFrame per record set. If not, we fall back to reading records through the generic iterator and store them under a synthetic key for convenience in the rest of the notebook.

This lets the analysis remain dynamic and reusable.

In [ ]:
dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        print(f"Loading record set: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"  Loaded {len(dataframes[record_set_id])} records")
        except Exception as e:
            print(f"  Failed to load {record_set_id}: {e}")
else:
    fallback_record_set_id = "@id:not_exposed_in_metadata"
    print("Loading records via dataset.records() fallback...")
    try:
        fallback_records = list(dataset.records())
        dataframes[fallback_record_set_id] = pd.DataFrame(fallback_records)
        print(f"Loaded {len(dataframes[fallback_record_set_id])} records into fallback DataFrame")
        sample_record_set_id = fallback_record_set_id
    except Exception as e:
        print(f"Fallback loading failed: {e}")

In [ ]:
print("Available DataFrames:")
for rs_id, df in dataframes.items():
    print(f"- {rs_id}: shape={df.shape}")

active_record_set_id = sample_record_set_id if sample_record_set_id in dataframes else (list(dataframes.keys())[0] if dataframes else None)
print(f"\nActive record set for analysis: {active_record_set_id}")

In [ ]:
if active_record_set_id is not None:
    active_df = dataframes[active_record_set_id].copy()
    print("Columns in the active DataFrame:")
    print(active_df.columns.tolist())
    display(active_df.head())
else:
    print("No DataFrame is available for analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data-processing steps to prepare the occurrence data for analysis.

Based on the dataset description, likely fields include:
- `Checked`
- `Country`
- `Date`
- `Family`
- `Genus`
- `Holotype`
- `Latitude`
- `Longitude`
- `Museum_number`
- `Reference`
- `Species`
- `Year`

Since actual loaded column names may vary, we detect columns dynamically using variable-driven logic.

In [ ]:
if active_record_set_id is not None:
    df = dataframes[active_record_set_id].copy()

    # Candidate field @ids / names expected from the dataset description
    candidate_fields = {
        'checked': ['Checked'],
        'country': ['Country'],
        'date': ['Date'],
        'family': ['Family'],
        'genus': ['Genus'],
        'holotype': ['Holotype'],
        'latitude': ['Latitude'],
        'longitude': ['Longitude'],
        'museum_number': ['Museum_number'],
        'reference': ['Reference'],
        'species': ['Species'],
        'year': ['Year']
    }

    resolved_fields = {}
    for label, candidates in candidate_fields.items():
        resolved_fields[label] = next((c for c in candidates if c in df.columns), None)

    print("Resolved analysis fields:")
    pprint(resolved_fields)
else:
    print("No active DataFrame found.")

In [ ]:
if active_record_set_id is not None:
    df = dataframes[active_record_set_id].copy()

    # Convert expected numeric columns safely
    for numeric_col in [resolved_fields.get('latitude'), resolved_fields.get('longitude'), resolved_fields.get('year')]:
        if numeric_col is not None:
            df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')

    print("Data types after conversion:")
    print(df.dtypes)

    print("\nMissing-value summary:")
    missing_summary = df.isna().sum().sort_values(ascending=False)
    display(missing_summary.to_frame(name='missing_count'))
else:
    print("No active DataFrame found.")

In [ ]:
# Example filtering and normalization using a numeric field, following the template style.
if active_record_set_id is not None:
    df = dataframes[active_record_set_id].copy()
    numeric_field = resolved_fields.get('year')
    group_field = resolved_fields.get('country')

    if numeric_field is not None:
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        # A simple threshold example: keep records after year 1900.
        threshold = 1900
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)}")
        display(filtered_df.head())

        if filtered_df[numeric_field].notna().sum() > 1 and filtered_df[numeric_field].std() not in [0, np.nan]:
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean())
                / filtered_df[numeric_field].std()
            )
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        else:
            print(f"Skipping normalization for {numeric_field} because there are too few valid values or zero variance.")

        if group_field is not None and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['count', 'mean', 'min', 'max']).sort_values('count', ascending=False)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head(10))
    else:
        print("No suitable numeric field was found for the threshold/normalization example.")
else:
    print("No active DataFrame found.")

In [ ]:
# Geographic cleaning example using latitude and longitude
if active_record_set_id is not None:
    df = dataframes[active_record_set_id].copy()
    latitude_field = resolved_fields.get('latitude')
    longitude_field = resolved_fields.get('longitude')

    if latitude_field is not None and longitude_field is not None:
        df[latitude_field] = pd.to_numeric(df[latitude_field], errors='coerce')
        df[longitude_field] = pd.to_numeric(df[longitude_field], errors='coerce')

        geo_df = df.dropna(subset=[latitude_field, longitude_field]).copy()
        print(f"Records with both coordinates present: {len(geo_df)}")

        # Remove impossible coordinate values
        geo_df = geo_df[
            geo_df[latitude_field].between(-90, 90)
            & geo_df[longitude_field].between(-180, 180)
        ].copy()

        print(f"Records remaining after coordinate range filtering: {len(geo_df)}")
        display(geo_df[[latitude_field, longitude_field]].describe())
    else:
        print("Latitude/Longitude fields were not found in the loaded data.")
else:
    print("No active DataFrame found.")

## 5. Visualization
Visualize distributions and relationships in the dataset.

The following plots are generated only when the corresponding fields are available.

In [ ]:
if active_record_set_id is not None:
    df = dataframes[active_record_set_id].copy()

    year_field = resolved_fields.get('year')
    country_field = resolved_fields.get('country')
    family_field = resolved_fields.get('family')
    latitude_field = resolved_fields.get('latitude')
    longitude_field = resolved_fields.get('longitude')

    fig_count = 0

    if year_field is not None:
        df[year_field] = pd.to_numeric(df[year_field], errors='coerce')
        year_plot_df = df.dropna(subset=[year_field]).copy()
        if not year_plot_df.empty:
            plt.figure(figsize=(10, 5))
            sns.histplot(year_plot_df[year_field], bins=30, kde=True)
            plt.title(f"Distribution of {year_field}")
            plt.xlabel(year_field)
            plt.ylabel("Count")
            plt.show()
            fig_count += 1

    if country_field is not None and country_field in df.columns:
        top_countries = df[country_field].value_counts(dropna=True).head(10)
        if not top_countries.empty:
            plt.figure(figsize=(10, 5))
            sns.barplot(x=top_countries.values, y=top_countries.index, orient='h')
            plt.title(f"Top 10 {country_field} values")
            plt.xlabel("Record count")
            plt.ylabel(country_field)
            plt.show()
            fig_count += 1

    if family_field is not None and family_field in df.columns:
        top_families = df[family_field].value_counts(dropna=True).head(10)
        if not top_families.empty:
            plt.figure(figsize=(10, 5))
            sns.barplot(x=top_families.values, y=top_families.index, orient='h')
            plt.title(f"Top 10 {family_field} values")
            plt.xlabel("Record count")
            plt.ylabel(family_field)
            plt.show()
            fig_count += 1

    if latitude_field is not None and longitude_field is not None:
        df[latitude_field] = pd.to_numeric(df[latitude_field], errors='coerce')
        df[longitude_field] = pd.to_numeric(df[longitude_field], errors='coerce')
        geo_plot_df = df.dropna(subset=[latitude_field, longitude_field]).copy()
        geo_plot_df = geo_plot_df[
            geo_plot_df[latitude_field].between(-90, 90) & geo_plot_df[longitude_field].between(-180, 180)
        ]
        if not geo_plot_df.empty:
            plt.figure(figsize=(8, 6))
            sns.scatterplot(data=geo_plot_df, x=longitude_field, y=latitude_field, s=20, alpha=0.5)
            plt.title("Geographic distribution of occurrence records")
            plt.xlabel(longitude_field)
            plt.ylabel(latitude_field)
            plt.show()
            fig_count += 1

    if fig_count == 0:
        print("No plots were generated because the expected fields were not available in the loaded DataFrame.")
else:
    print("No active DataFrame found.")

## 6. Conclusion
This notebook demonstrated how to:

- Load a Croissant dataset with `mlcroissant`
- Inspect dataset-level metadata
- Discover and reference record sets and fields using `@id` values when exposed
- Extract records into pandas DataFrames
- Perform basic exploratory data analysis on temporal, geographic, and categorical fields
- Create simple visualizations for occurrence patterns

### Key observations from the metadata
- The dataset describes African bat occurrence records spanning a long temporal range.
- It includes taxonomic, specimen, geographic, and reference-linked information.
- The package summary highlights important missingness in fields such as `Checked`, `Date`, `Holotype`, `Museum_number`, and some coordinates.
- The metadata also warns about sampling, temporal, verification, and documentation biases.

### Final note
If the Croissant source is updated to expose explicit record sets and field definitions, the same notebook structure can be reused directly by replacing the dynamically discovered `@id` values with the exact record set and field `@id`s shown in the metadata.